# KronLLM — tests, diagnostics and generation

**KronLLM** is a language model in which every weight matrix (Q, K, V, O, FFN)
is not a dense table but a **Kronecker decomposition**

$$W_{i,l} = s\,I + \sum_{k=1}^{K} A_{i,l,k} \otimes B_{i,l,k},
\qquad A\in\mathbb{R}^{n_1\times n_1},\; B\in\mathbb{R}^{n_2\times n_2},$$

which is then composed over $L$ channels: $P_i = \prod_{l=1}^{L} W_{i,l}$.

The project started with a hard budget of **1 000 000 parameters**, respected by
`E0`/`E2`/`Q1`. It then turned out that the real constraint is the **6 GB of
VRAM on a GTX 1660 Ti**, not the parameter count: the final model `Q2`
deliberately exceeds the budget because it occupies the same ~4.2 GB as Q1
while doing three times the compute per token.

Inside this notebook:

| section | contents |
|---|---|
| 1 | parameter budget and FLOPs (embedding / matrices / norms balance) |
| 2 | implementation invariants: `kron_apply` ≡ dense matrix, bank ≡ per-module path, causality, tied head |
| 3 | training curves in **bits per byte** — one metric for byte and BPE tokenisation |
| 4 | diagnostics: effective rank vs depth, spectra, $\tau$ / $\gamma$, $\sigma(\theta)$, attention entropy |
| 5 | generation at $T=1.0$ and $top_k\in\{5,8,10\}$ |
| 6 | supervised fine-tuning (S1) and int8/int4 quantisation |
| 7 | final comparison and the verdict |

**The models being compared**

* `E0` — dense control: same corpus, same 2 500 steps, 969 856 parameters,
  byte level (an upper bound for "what one million parameters buys");
* `E2` — byte-level Kronecker baseline ($V=256$, 190 108 parameters, 2 500 steps);
* `Q1` — the first quality branch: byte-level BPE with 1024 tokens (**3.05
  bytes/token** — the "three bytes at a time" idea with learned boundaries),
  $d=192$, 20 layers, $K=5$, factorised tied embedding, QK-norm (926 468
  parameters). Stopped at step 4 200 (0.976 bpb) to give the GPU to a larger
  model;
* `Q2` — the final model for 6 GB: BPE-1024, $d=320$, 24 layers, $K=6$,
  $n_1\times n_2 = 16\times 20$, 2 079 992 parameters, 10 000 steps;
* `D1` — dense control on the *same* BPE tokenizer and corpus, so the price of
  the Kronecker parameterisation is measurable in bits per byte;
* `S1` — Q2 fine-tuned on Dolly-15k Q/A pairs, plus its int8/int4 checkpoints.

## 0. Setup

Everything runs on the GPU when one is available. The notebook can be started
from the repository root or from `notebooks/` — the root is found automatically.

In [ ]:
import html, json, math, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "kronllm").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)          # configs use paths relative to the repository root

from kronllm.data import open_dataset
from kronllm.diagnostics import (eff_rank_heatmap, sigma_theta_report,
                                 spectral_report, spearman)
from kronllm.generate import generate_texts
from kronllm.kron import kron_apply, kron_materialize
from kronllm.model import KronTransformer
from kronllm.quantization import load_any_checkpoint
from kronllm.tokenizer import bits_per_byte, load_tokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

COLORS = {"E2": "#7aa2f7", "Q1": "#bb9af7", "Q2": "#9ece6a", "D1": "#e0af68",
          "S1": "#f7768e"}
LABELS = {"E2": "E2 · byte · 190K", "Q1": "Q1 · BPE · 926K",
          "Q2": "Q2 · BPE · 2.08M", "D1": "D1 · dense · 1.07M",
          "S1": "S1 · Q2 + Dolly Q/A"}

plt.rcParams.update({
    "figure.facecolor": "#1a1b26", "savefig.facecolor": "#1a1b26",
    "axes.facecolor": "#16161e", "axes.edgecolor": "#3b4261",
    "axes.labelcolor": "#c0caf5", "text.color": "#c0caf5",
    "xtick.color": "#a9b1d6", "ytick.color": "#a9b1d6",
    "grid.color": "#2a2e42", "axes.grid": True, "grid.alpha": 0.55,
    "font.size": 11, "figure.dpi": 110, "axes.titleweight": "bold",
    "axes.prop_cycle": matplotlib.cycler(color=[
        "#7aa2f7", "#bb9af7", "#9ece6a", "#e0af68", "#f7768e", "#7dcfff"]),
})


def style(ax, title=None, xlabel=None, ylabel=None):
    if title:
        ax.set_title(title, pad=10)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    return ax


def load_run(name, checkpoint="best.pt"):
    '''Load a run into memory; works for float and quantised checkpoints.'''
    d = ROOT / "runs" / name
    path = d / checkpoint
    if not path.exists():
        path = d / "last.pt"
    cfg, state = load_any_checkpoint(path)
    if cfg is None:
        cfg = yaml.safe_load((d / "config.yaml").read_text(encoding="utf-8"))
    model = KronTransformer(cfg)
    model.load_state_dict(state)
    model.to(DEVICE).eval()
    return {"name": name, "dir": d, "model": model, "cfg": cfg,
            "tok": load_tokenizer(cfg),
            "step": torch.load(path, map_location="cpu", weights_only=False).get("step"),
            "params": sum(p.numel() for p in model.parameters())}


RUNS = {}
for _name in ("E2", "Q1", "Q2", "D1", "S1"):
    if (ROOT / "runs" / _name).exists():
        RUNS[_name] = load_run(_name)

print(f"device = {DEVICE}  |  torch {torch.__version__}")
for _n, _r in RUNS.items():
    print(f"{_n}: {_r['params']:>7,} parameters · vocab {_r['tok'].vocab_size:>4} · "
          f"{_r['tok'].bytes_per_token:.2f} bytes/token · step {_r['step']}")

## 1. Parameter budget

The historical requirement was **≤ 1 000 000 parameters with no "selfish"
block**; `E0`/`E2`/`Q1` follow it. `Q2` lifts that limit on purpose: the
bottleneck is 6 GB of VRAM, not the parameter count. The balance is preserved:
matrices ~85–90 %, embedding ~8–13 %, norms ~1 %.

The factorised embedding (ALBERT-style) is what frees the budget: instead of a
$V\times d$ table, $V\times r$ and $r\times d$ are stored. With $V=1024$,
$r=96$, $d=192$ that is $116\,736$ parameters instead of $196\,608$; for Q2 with
$r=128$, $d=320$ it is $172\,032$ instead of $327\,680$. The head is **tied**
to the embedding and costs nothing extra: $\text{logits} = h\,U^\top E^\top$.

In [ ]:
def budget(model):
    total = sum(p.numel() for p in model.parameters())
    emb = sum(p.numel() for n, p in model.named_parameters()
              if "embed" in n or "lm_head" in n)
    nrm = sum(p.numel() for n, p in model.named_parameters() if "norm" in n)
    mat = model.n_params_matrix
    return {"total": total, "matrices": mat, "embed": emb, "norms": nrm,
            "other": total - mat - emb - nrm}


rows = []
for name, r in RUNS.items():
    m, cfg = r["model"], r["cfg"]
    b = budget(m)
    fl = m.flops_per_token(seq_len=m.seq_len, batch_size=64)
    bpt = r["tok"].bytes_per_token
    rows.append({
        "model": name,
        "tokenizer": r["tok"].name,
        "vocab": r["tok"].vocab_size,
        "d": m.d,
        "layers": m.n_layers,
        "K": cfg["model"]["kron"]["K"],
        "total": b["total"],
        "matrices, %": 100 * b["matrices"] / b["total"],
        "embedding, %": 100 * b["embed"] / b["total"],
        "norms, %": 100 * b["norms"] / b["total"],
        "bytes/token": bpt,
        "MFLOP/token": fl / 1e6,
        "MFLOP/byte": fl / bpt / 1e6,
    })

df_budget = pd.DataFrame(rows).set_index("model")
display(df_budget.style.format({
    "total": "{:,}", "matrices, %": "{:.1f}", "embedding, %": "{:.1f}",
    "norms, %": "{:.2f}", "bytes/token": "{:.2f}",
    "MFLOP/token": "{:.2f}", "MFLOP/byte": "{:.2f}",
}).set_caption("Parameter budget: the 1M design limit and the final 6 GB model"))

under_1m = [n for n, r in RUNS.items() if budget(r["model"])["total"] <= 1_000_000]
over_1m = [n for n, r in RUNS.items() if budget(r["model"])["total"] > 1_000_000]
print(f"within the 1M design budget: {', '.join(under_1m) or '-'}")
print(f"deliberately above 1M (the limit is 6 GB of VRAM): {', '.join(over_1m) or '-'}")
assert all(budget(r["model"])["total"] < 8_000_000 for r in RUNS.values())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 1.2 + 0.75 * len(RUNS)))
parts = [("matrices", "matrices", "#7aa2f7"),
         ("embed", "embedding", "#bb9af7"),
         ("norms", "norms", "#9ece6a"),
         ("other", "other", "#565f89")]
left = np.zeros(len(RUNS))
for key, label, color in parts:
    vals = np.array([budget(r["model"])[key] for r in RUNS.values()])
    ax.barh(list(RUNS), vals, left=left, color=color, label=label, height=0.55)
    for i, (v, l) in enumerate(zip(vals, left)):
        if v > 25_000:
            ax.text(l + v / 2, i, f"{v/1000:.0f}k", ha="center", va="center",
                    fontsize=9, color="#1a1b26", fontweight="bold")
    left += vals
ax.axvline(1_000_000, color="#f7768e", ls="--", lw=1.4, label="1M design limit")
_tot_max = max(budget(r["model"])["total"] for r in RUNS.values())
ax.set_xlim(0, _tot_max * 1.08)
ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
    lambda v, _: f"{v/1000:.0f}k"))
ax.set_title("Distribution of the parameter budget", pad=30)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(ncol=5, loc="lower left", bbox_to_anchor=(0, 1.01), frameon=False,
          fontsize=9)
plt.tight_layout()
plt.show()

## 2. Implementation invariants

The mathematics must match the code, otherwise every plot below is meaningless.
Six claims are checked:

1. `kron_materialize(A, B)` equals `torch.kron(A, B)`;
2. `kron_apply(A, B, x)` is exactly multiplication by the dense $A\otimes B$;
3. the batched bank (`KronBank`) produces the same logits as the per-block $P_i$ path;
4. causality: changing future tokens does not change past logits;
5. the factorised tied head computes the formula it claims to;
6. the tokenizer is invertible: `decode(encode(text)) == text`.

In [ ]:
checks = []


def check(name, ok, detail=""):
    checks.append({"test": name, "result": "PASS" if ok else "FAIL", "detail": detail})


# 1-2. Kronecker operations
A, B = torch.randn(6, 6), torch.randn(8, 8)
Wd = kron_materialize(A, B)
check("kron_materialize == torch.kron",
      torch.allclose(Wd, torch.kron(A, B), atol=1e-5),
      f"max|delta| = {(Wd - torch.kron(A, B)).abs().max():.2e}")

x = torch.randn(3, 6, 8)
y_apply = kron_apply(A, B, x).reshape(3, 48)
y_dense = x.reshape(3, 48) @ Wd.T
check("kron_apply == (A kron B) x",
      torch.allclose(y_apply, y_dense, atol=1e-5),
      f"max|delta| = {(y_apply - y_dense).abs().max():.2e}")

# 3-5. model invariants, checked on whichever run is available
_ref = RUNS.get("Q2") or RUNS.get("Q1")
if _ref is not None:
    m = _ref["model"]
    cfg = _ref["cfg"]
    xb, yb = next(iter(open_dataset(cfg, "val").iter_blocks(2, DEVICE)))

    lg_bank = m(xb)["logits"].clone()
    banks = m._banks
    m._banks = []
    m._prelude_cache = None          # invalidate the eval-mode prelude cache
    lg_seq = m(xb)["logits"].clone()
    m._banks = banks
    m._prelude_cache = None
    check("bank (batched prelude) == per-module path",
          torch.allclose(lg_bank, lg_seq, atol=2e-2),
          f"max|delta| = {(lg_bank - lg_seq).abs().max():.2e}")

    x2 = xb.clone()
    x2[:, -8:] = torch.randint_like(x2[:, -8:], 0, 512)
    cut = xb.shape[1] - 8
    l1 = m(xb)["logits"][:, :cut]
    l2 = m(x2)["logits"][:, :cut]
    check("causality (the future does not change the past)",
          torch.allclose(l1, l2, atol=1e-4),
          f"max|delta| = {(l1 - l2).abs().max():.2e}")

    h = torch.randn(2, 5, m.d, device=DEVICE)
    ref = (h @ m.embed_up.weight) @ m.embed.weight.t()
    check("tied factorised head == h U E^T",
          torch.allclose(m._logits(h), ref, atol=1e-4),
          f"shape {tuple(m._logits(h).shape)}")

# 6. tokenizer
for name, r in RUNS.items():
    probe = "Once upon a time, there was a little girl named Lily."
    check(f"decode(encode(text)) == text [{name}]",
          r["tok"].decode(r["tok"].encode(probe)) == probe,
          f"{len(r['tok'].encode(probe))} tokens")

display(pd.DataFrame(checks).set_index("test"))
assert all(c["result"] == "PASS" for c in checks)
print("all invariants hold")

## 3. Training curves

The loss is normalised per **byte of text**, not per token:

$$\mathrm{bpb} = \frac{\mathcal{L}_{\text{nats/token}}}{\text{bytes/token}\cdot\ln 2}.$$

This is the only fair way to compare the byte model (1 token = 1 byte) with BPE
(1 token ≈ 3 bytes). For the byte model bpb equals the loss in bits.

`E0` (dense, 970K, byte level) and `D1` (dense, 1.07M, BPE) are the controls:
they answer what a dense model of the same size achieves on the same corpus,
and they are what the Kronecker parameterisation is priced against.

In [ ]:
def load_log(name):
    r = RUNS[name]
    df = pd.read_csv(r["dir"] / "log.csv")
    bpt = r["tok"].bytes_per_token
    if "bpb" not in df.columns or df["bpb"].isna().all():
        df["bpb"] = df["loss"].map(lambda v: bits_per_byte(v, bpt))
    return df


LOGS = {n: load_log(n) for n in RUNS}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for name, df in LOGS.items():
    tr = df[df.split == "train"]
    va = df[df.split == "val"]
    c = COLORS[name]
    axes[0].plot(tr.step, tr.loss.rolling(20, min_periods=1).mean(),
                 color=c, lw=1.1, alpha=0.55)
    if len(va):
        axes[0].plot(va.step, va.loss, color=c, lw=2.0, marker="o", ms=3,
                     label=f"{name} · val {va.loss.iloc[-1]:.3f}")
        axes[1].plot(va.step, va.bpb, color=c, lw=2.0, marker="o", ms=3,
                     label=f"{name} · {va.bpb.iloc[-1]:.3f} bpb · "
                           f"{int(va.step.iloc[-1])} steps")
    else:
        axes[0].plot([], [], color=c, label=f"{name} · no val log")
        axes[1].plot([], [], color=c, label=f"{name} · no val log")

# dense controls: E0 on bytes, D1 on the same BPE corpus as Q1
for _ctl, _color, _ls in (("E0", "#e0af68", "--"), ("D1", "#f7768e", ":")):
    _path = ROOT / "runs" / _ctl / "log.csv"
    if not _path.exists():
        continue
    _df = pd.read_csv(_path)
    _va = _df[_df.split == "val"]
    if not len(_va):
        continue
    _bpt = 1.0 if _ctl == "E0" else None
    if _bpt is None:
        _cfg = yaml.safe_load((ROOT / "runs" / _ctl / "config.yaml")
                              .read_text(encoding="utf-8"))
        _bpt = load_tokenizer(_cfg).bytes_per_token
    _bpb = _va.loss.map(lambda v: bits_per_byte(v, _bpt))
    axes[1].plot(_va.step, _bpb, color=_color, ls=_ls, lw=1.6,
                 label=f"{_ctl} dense · {_bpb.iloc[-1]:.3f} bpb")

axes[0].axhline(math.log(256), color="#565f89", ls=":", lw=1)
axes[0].text(0.02, math.log(256) + 0.03, "ln 256 - a random byte",
             transform=axes[0].get_yaxis_transform(), fontsize=8, color="#565f89")
style(axes[0], "Cross-entropy", "step", "nats/token")
axes[0].legend(frameon=False)
style(axes[1], "Bits per byte (the common metric)", "step", "bpb")
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

finals = {k: v.bpb.iloc[-1] for k, v in LOGS.items() if len(v[v.split == "val"])}
if finals:
    best = min(finals.items(), key=lambda kv: kv[1])
    print(f"best model by bpb: {best[0]} - {best[1]:.3f} bits/byte")

## 4. Diagnostics

What to look at:

* **effective rank** $\mathrm{er}(D)=\exp H(\sigma/\sum\sigma)$ — how many
  directions actually carry the innovation of each channel; one Kronecker
  summand is capped at $\min(n_1^2,n_2^2)$, while the composition can reach $d$;
* **spectra** of $P_i$ — does the composition recover rank, and how close is it
  to the identity matrix (the anchor $s$);
* **$\tau$ and $\gamma$** — the scales of the matrix output and of the gate: if
  they grow with depth, the model is compensating for attenuation;
* **$\sigma(\theta)$** — the spread of the learned memory factors; too small
  means the factors do nothing;
* **attention entropy** — has attention collapsed onto a single position?

In [ ]:
SPEC, STATS = {}, {}
for name, r in RUNS.items():
    ds = open_dataset(r["cfg"], "val")
    xb, yb = next(iter(ds.iter_blocks(4, DEVICE)))
    res = r["model"](xb, yb, collect=True)
    SPEC[name] = spectral_report(r["model"], res["trace"])
    STATS[name] = res["stats"]

summary = pd.DataFrame({n: {k: v for k, v in SPEC[n]["summary"].items()
                            if isinstance(v, (int, float))} for n in SPEC})
keep = ["eff_rank_D", "eff_rank_P", "eff_rank_excess", "eff_rank_W",
        "eff_rank_spearman", "eff_rank_growth", "tau_mean", "gate_mean",
        "cos_prev_P_mean", "cond_max"]
display(summary.loc[[k for k in keep if k in summary.index]]
        .style.format("{:.3f}").set_caption("Spectral diagnostics summary"))

In [ ]:
ROLES = ["q", "k", "v", "o", "f", "fg"]
fig, axes = plt.subplots(1, len(RUNS), figsize=(4.4 * len(RUNS), 4.6), sharex=True)
axes = np.atleast_1d(axes)
for ax, name in zip(axes, RUNS):
    spec, model = SPEC[name], RUNS[name]["model"]
    layers = sorted({row["layer"] for row in spec["rows"]})
    for role in ROLES:
        ys = [np.mean([row["eff_rank_D"] for row in spec["rows"]
                       if row["layer"] == i and row["role"] == role] or [np.nan])
              for i in layers]
        if np.all(np.isnan(ys)):
            continue
        ax.plot(layers, ys, marker="o", ms=3, lw=1.6, label=role)
    ax.set_ylim(0, model.d)
    style(ax, f"{name} · effective rank of the innovation $D$", "layer", "er(D)")
    ax.legend(ncol=3, frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 3.6))
for name, spec in SPEC.items():
    layers = sorted({row["layer"] for row in spec["rows"]})
    ys = [np.mean([row["eff_rank_excess"] for row in spec["rows"]
                   if row["layer"] == i]) for i in layers]
    ax.plot(layers, ys, marker="o", ms=3, lw=2, color=COLORS[name],
            label=f"{name} · rho={spearman(layers, ys):+.2f}")
style(ax, "Rank of the composition without the anchor: $P - s^L I$", "layer", "er")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(RUNS), figsize=(4.6 * len(RUNS), 4.2))
axes = np.atleast_1d(axes)
for ax, name in zip(axes, RUNS):
    spec = SPEC[name]
    layers = sorted({row["layer"] for row in spec["rows"]})
    for role, color in zip(["q", "f"], ["#7aa2f7", "#e0af68"]):
        rows = [(i, row) for i in layers for row in spec["rows"]
                if row["layer"] == i and row["role"] == role and row["channel"] == 0]
        if not rows:
            continue
        ax.plot([i for i, _ in rows], [r["tau"] for _, r in rows],
                color=color, lw=2, marker="o", ms=3, label=f"tau[{role}]")
        ax.plot([i for i, _ in rows], [r["gate"] for _, r in rows],
                color=color, lw=1.2, ls="--", alpha=0.75, label=f"gamma[{role}]")
    style(ax, f"{name} · scales tau and gate gamma", "layer", "value")
    ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(RUNS), figsize=(6 * len(RUNS), 3.8), squeeze=False)
for ax, (name, r) in zip(axes[0], RUNS.items()):
    sig = sigma_theta_report(r["model"])
    flat = [v for rows in sig["sigma_theta_per_type"].values()
            for row in rows for v in row]
    ax.hist(flat, bins=30, color=COLORS[name], alpha=0.85)
    ax.axvline(sig["sigma_theta_mean"], color="#e0af68", ls="--", lw=1.2,
               label=f"mean {sig['sigma_theta_mean']:.3f}")
    style(ax, f"{name} · distribution of sigma(theta)", "sigma(theta)", "count")
    ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

if RUNS:
    _big = max(RUNS, key=lambda n: budget(RUNS[n]["model"])["total"])
    sig = sigma_theta_report(RUNS[_big]["model"])
    grid = np.array(sig["sigma_theta_per_type"]["q"])
    fig, ax = plt.subplots(figsize=(6.5, 7))
    im = ax.imshow(grid, aspect="auto", origin="lower", cmap="magma")
    ax.set_xlabel("channel $l$")
    ax.set_ylabel("layer")
    ax.set_title(f"{_big} · $\\sigma(\\theta)$ for Q: layer x channel", pad=10)
    fig.colorbar(im, ax=ax, label="$\\sigma(\\theta)$")
    plt.tight_layout()
    plt.show()
    print(f"Var sigma(theta) = {sig['sigma_theta_var']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))
for name, r in RUNS.items():
    st = STATS[name]
    layers = sorted({int(k.split(".")[0][1:]) for k in st if k.startswith("L")})
    ent = [st.get(f"L{i}.attn_entropy", np.nan) for i in layers]
    drift = [st.get(f"L{i}.q_drift", np.nan) for i in layers]
    axes[0].plot(layers, ent, marker="o", ms=3, lw=2, color=COLORS[name], label=name)
    axes[1].plot(layers, drift, marker="o", ms=3, lw=2, color=COLORS[name], label=name)
style(axes[0], "Attention entropy per layer", "layer", "H(attn), nats")
axes[0].legend(frameon=False)
style(axes[1], "Drift of q relative to the block input", "layer", "norm(q - h) / norm(q)")
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown("### What the model predicts next"))


def tok_label(tok, i: int) -> str:
    s = tok.decode([int(i)])
    return repr(s) if s and all(ch.isprintable() for ch in s) else f"<0x{int(i):02x}>"


PROBE = "Once upon a time there was a little"
for name, r in RUNS.items():
    tok = r["tok"]
    ids = tok.encode(PROBE)
    x = torch.tensor([ids], device=DEVICE)
    logits = r["model"](x)["logits"][0, -1].float()
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, 8)
    rows = [{"token": tok_label(tok, int(i)), "probability": float(p)}
            for p, i in zip(top.values, top.indices)]
    display(Markdown(f"**{name}** · context `{PROBE}` · {len(ids)} tokens"))
    display(pd.DataFrame(rows).style.format({"probability": "{:.3f}"}))

In [ ]:
import torch.nn.functional as F

def position_profile(cfg, model, n_batches: int = 8):
    ds = open_dataset(cfg, "val")
    bpt = load_tokenizer(cfg).bytes_per_token
    total, count = None, 0
    for i, (x, y) in enumerate(ds.iter_blocks(16, DEVICE)):
        if i >= n_batches:
            break
        logits = model(x)["logits"].float()
        ls = F.cross_entropy(logits.view(-1, logits.size(-1)),
                             y.reshape(-1), reduction="none").view(y.shape)
        total = ls.sum(0) if total is None else total + ls.sum(0)
        count += ls.shape[0]
    return (total / max(count, 1)).cpu().numpy() / bpt / math.log(2)


fig, ax = plt.subplots(figsize=(9, 4))
for name, r in RUNS.items():
    prof = position_profile(r["cfg"], r["model"])
    x = np.linspace(0, 100, len(prof))
    smooth = pd.Series(prof).rolling(9, min_periods=1, center=True).mean()
    ax.plot(x, smooth, color=COLORS[name], lw=2, label=name)
    ax.fill_between(x, smooth, alpha=0.12, color=COLORS[name])
ax.set_xlabel("position in the context, % of length")
ax.set_ylabel("bpb")
style(ax, "How many bits a byte costs depending on its position")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print("a falling bpb from the start to the end of the context is the payoff from context;")
print("if the curve is flat, the model is not using its history.")

## 5. Generation

Rules for a small model: **$T=1.0$** (do not "tune" the distribution) and
**$top_k=5\text{–}10$** (cut the long tail the model cannot score anyway). Below
is the same model at three values of $top_k$.

In [ ]:
def card(title, text, color="#7aa2f7"):
    return Markdown(
        f'<div style="border-left:4px solid {color};background:#16161e;'
        f'padding:10px 14px;margin:8px 0;border-radius:6px">'
        f'<div style="color:#7dcfff;font-size:11px;letter-spacing:.08em;'
        f'text-transform:uppercase;margin-bottom:6px">{html.escape(title)}</div>'
        f'<div style="color:#c0caf5;font-family:Georgia,serif;font-size:15px;'
        f'line-height:1.55;white-space:pre-wrap">{html.escape(text)}</div></div>')


PROMPTS = ["Once upon a time", "The little cat", "One day, a girl named Lily"]
_topk_model = "Q2" if "Q2" in RUNS else ("Q1" if "Q1" in RUNS else None)
if _topk_model:
    r = RUNS[_topk_model]
    for k in (5, 8, 10):
        display(Markdown(f"### {_topk_model} · $T=1.0$, $top_k={k}$"))
        texts = generate_texts(r["model"], PROMPTS, max_new_tokens=220,
                               temperature=1.0, top_k=k, n_samples=len(PROMPTS),
                               seed=7, device=DEVICE, tokenizer=r["tok"])
        for prompt, text in zip(PROMPTS, texts):
            display(card(prompt, text, COLORS[_topk_model]))

In [ ]:
if len(RUNS) > 1:
    display(Markdown("### The same prompt across all models"))
    prompt = "Once upon a time"
    for name, r in RUNS.items():
        text = generate_texts(r["model"], [prompt], max_new_tokens=220,
                              temperature=1.0, top_k=8, n_samples=1, seed=11,
                              device=DEVICE, tokenizer=r["tok"])[0]
        display(card(f"{LABELS[name]} · T=1.0, top_k=8", text, COLORS[name]))

In [ ]:
display(Markdown("### Generation speed"))
speed = []
for name, r in RUNS.items():
    m, tok = r["model"], r["tok"]
    idx = torch.tensor([tok.encode("Once upon a time")], device=DEVICE)
    m.generate(idx, max_new_tokens=8, temperature=1.0, top_k=8)   # warm-up
    t0 = time.perf_counter()
    m.generate(idx, max_new_tokens=64, temperature=1.0, top_k=8)
    dt = time.perf_counter() - t0
    speed.append({"model": name, "tokens/s": 64 / dt,
                  "bytes/s": 64 / dt * tok.bytes_per_token,
                  "ms/token": 1000 * dt / 64})
display(pd.DataFrame(speed).set_index("model").style.format(
    {"tokens/s": "{:.1f}", "bytes/s": "{:.0f}", "ms/token": "{:.1f}"}))

## 6. Supervised fine-tuning (S1)

A 2M-parameter model cannot become a general assistant: what it can learn is the
Q/A **format** and the questions whose answers live in a small closed set. S1
makes that explicit.

* data: `databricks/databricks-dolly-15k`, Q/A categories only (7 706 rows, of
  which 4 882 fit the 192-token context);
* template: `### Question:` / `### Answer:` with the answer terminated by EOT;
* training: Q2 weights (`training.init_from`), a fresh optimizer, 400 steps at
  lr 1e-4.

The same template is used below, at generation time.

In [ ]:
QA_PROMPTS = [
    "### Question:\nWhat is the capital of France?\n### Answer:\n",
    "### Question:\nName three colours.\n### Answer:\n",
    "### Question:\nWho wrote Romeo and Juliet?\n### Answer:\n",
]
if "S1" in RUNS:
    r = RUNS["S1"]
    for prompt in QA_PROMPTS:
        text = generate_texts(r["model"], [prompt], max_new_tokens=80,
                              temperature=1.0, top_k=8, n_samples=1, seed=3,
                              device=DEVICE, tokenizer=r["tok"])[0]
        display(card(prompt.replace("\n", " "), text, COLORS["S1"]))
    print("Note: the answers are memorised patterns from a 2M-parameter model,")
    print("not knowledge. Expect the format to hold and the content to drift.")
else:
    print("runs/S1 not found - train it with: python -m kronllm.cli train --exp S1")

## 7. Quantisation (int8 / int4)

KronLLM has no `nn.Linear` layer, so `torch.ao.quantization` cannot see its
matrices. `kronllm/quantization.py` quantises the **stored tensors** (factors,
embeddings, tied head) with symmetric per-channel scales and dequantises them on
load, so the forward pass is unchanged and only the file size shrinks.

The table below is produced by:

```bash
python -m kronllm.cli quantize --run runs/S1 --bits 8 4
```

In [ ]:
_qpath = ROOT / "runs" / "S1" / "quantization.json"
if _qpath.exists():
    _q = json.loads(_qpath.read_text(encoding="utf-8"))
    rows = [{"checkpoint": "float32", "size, MB": _q["reports"][0]["size_before_mb"],
             "compression": 1.0, "val bpb": _q["fp32_bpb"], "delta bpb": 0.0}]
    for rep in _q["reports"]:
        rows.append({"checkpoint": f"int{rep['bits']}",
                     "size, MB": rep["size_after_mb"],
                     "compression": rep["compression"],
                     "val bpb": rep["val_bpb"],
                     "delta bpb": rep["delta_bpb"]})
    display(pd.DataFrame(rows).set_index("checkpoint").style.format({
        "size, MB": "{:.2f}", "compression": "{:.2f}x",
        "val bpb": "{:.4f}", "delta bpb": "{:+.4f}",
    }).set_caption("S1 checkpoint size and validation quality after quantisation"))
else:
    print("runs/S1/quantization.json not found - run the quantize command above.")

## 8. Summary

A single table across all runs, and the verdict.

In [ ]:
rows = []
for name, r in RUNS.items():
    res_path = r["dir"] / "results.json"
    res = json.loads(res_path.read_text(encoding="utf-8")) if res_path.exists() else {}
    b = budget(r["model"])
    bpt = res.get("bytes_per_token", r["tok"].bytes_per_token)
    va = LOGS[name][LOGS[name].split == "val"]
    # Q1 was stopped with SIGKILL to free the GPU, so it has no results.json:
    # fall back to the last val row of log.csv.
    val_loss = res.get("val_loss", float(va.loss.iloc[-1]) if len(va) else np.nan)
    val_loss = float(val_loss)
    val_ppl = res.get("val_ppl", math.exp(val_loss) if np.isfinite(val_loss) else np.nan)
    bpb = res.get("val_bpb")
    if bpb is None and np.isfinite(val_loss):
        bpb = bits_per_byte(val_loss, bpt)
    sec = res.get("sec_per_step")
    if sec is None:
        _s = LOGS[name].sec_per_step.dropna()
        sec = float(_s.tail(20).median()) if len(_s) else np.nan
    rows.append({
        "model": name,
        "parameters": b["total"],
        "matrices, %": 100 * b["matrices"] / b["total"],
        "val loss": val_loss,
        "val PPL (token)": val_ppl,
        "val bpb": bpb,
        "MFLOP/byte": r["model"].flops_per_token(
            seq_len=r["model"].seq_len, batch_size=64)
            / r["tok"].bytes_per_token / 1e6,
        "s/step": sec,
        "VRAM, GB": res.get("peak_vram_gb", np.nan),
        "steps": res.get("steps", int(LOGS[name].step.max())),
    })

df_final = pd.DataFrame(rows).set_index("model")
display(df_final.style.format({
    "parameters": "{:,}", "matrices, %": "{:.1f}", "val loss": "{:.4f}",
    "val PPL (token)": "{:.2f}", "val bpb": "{:.3f}", "MFLOP/byte": "{:.2f}",
    "s/step": "{:.3f}", "VRAM, GB": "{:.2f}", "steps": "{:.0f}",
}).set_caption("Final comparison"))

if "val bpb" in df_final and df_final["val bpb"].notna().any():
    fig, ax = plt.subplots(figsize=(6.8, 3.4))
    vals = df_final["val bpb"].dropna()
    ax.barh(vals.index, vals.values,
            color=[COLORS.get(i, "#7aa2f7") for i in vals.index], height=0.5)
    for i, v in enumerate(vals.values):
        ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=10)
    lim = vals.values.max()
    e0 = ROOT / "runs" / "E0" / "log.csv"
    if e0.exists():
        d0 = pd.read_csv(e0)
        d0 = d0[d0.split == "val"]
        if len(d0):
            v0 = bits_per_byte(float(d0.loss.iloc[-1]), 1.0)
            lim = max(lim, v0)
            ax.axvline(v0, color="#9ece6a", ls="--", lw=1.4,
                       label=f"E0 dense 970K · {v0:.3f} bpb")
            ax.legend(frameon=False, fontsize=9, loc="lower right")
    ax.set_xlim(0, lim * 1.12)
    ax.invert_yaxis()
    style(ax, "Bits per byte - lower is better", "bpb", None)
    plt.tight_layout()
    plt.show()

In [ ]:
vals = df_final["val bpb"].dropna()
if len(vals):
    name = str(vals.idxmin())
    best_bpb = float(vals.min())
    r = RUNS[name]
    b = budget(r["model"])
    gains = ""
    if len(vals) > 1:
        base = float(vals.drop(index=name).min())
        gains = (f"\n\nThe bpb gain over the runner-up is "
                 f"**{100 * (base - best_bpb) / base:+.1f} %**.")
    ref = ""
    e0 = ROOT / "runs" / "E0" / "log.csv"
    if e0.exists():
        d0 = pd.read_csv(e0)
        d0 = d0[d0.split == "val"]
        if len(d0):
            v0 = bits_per_byte(float(d0.loss.iloc[-1]), 1.0)
            ref = (f"\n\nAgainst the dense control **E0** (969 856 parameters, "
                   f"the same corpus at byte level, 2 500 steps): {v0:.3f} bpb, "
                   f"a gain of **{100 * (v0 - best_bpb) / v0:+.1f} %**.")
    _cfg = r["cfg"]
    _steps = int(LOGS[name].step.max())
    _tok_seen = _steps * _cfg["training"]["batch_size"] * _cfg["training"]["seq_len"]
    _epochs = _tok_seen / len(open_dataset(_cfg, "train").data)
    display(Markdown(f'''
## Verdict

The best model in terms of bits per byte is **{name}** ({b["total"]:,} parameters,
{b["matrices"] / b["total"] * 100:.0f} % of them in Kronecker matrices,
{r["tok"].bytes_per_token:.2f} bytes/token) at **{best_bpb:.3f} bpb**.
{gains}{ref}

Where the gain comes from:

1. **Tokenisation.** A 1024-token BPE compresses text 3.05x, so the same FLOPs
   per byte buy a three times longer context and three times fewer positions at
   which spelling has to be guessed. The byte frontier stays open, but the
   "three bytes at a time" come from learned merges rather than a fixed chunking
   (which would need a $256^3$ vocabulary).
2. **Capacity instead of a budget.** Q1 (926K) showed that a million parameters
   is not a ceiling: Q2 occupies the same ~4.2 GB of VRAM with
   {b["total"]:,} parameters, because memory is set by $N\\cdot d\\cdot batch$,
   not by the parameter count. The price is roughly three times the FLOPs per
   token.
3. **Normalisation.** QK-norm inside the block keeps $q,k$ on one scale, which
   removes logit drift and allows $lr\\approx 4\\cdot10^{-4}$ without divergence;
   pre-norm + RMSNorm stay in place.
4. **Training.** {_steps:,} steps ≈ {_epochs:.1f} epochs against E0's 0.45 - the
   cheapest source of quality there is.

Limitations: the effective rank of the innovation saturates at
$\\min(n_1^2,n_2^2)$ and the composition only partly recovers it, so further
quality is cheaper through data and steps than through $K$. A full pass over
TinyStories (2.1M stories) was never made: 200K stories are ~10 % of the corpus.
'''))